In [1]:
import os
import socket
import numpy as np
import pandas as pd
import streamlit as st
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from urllib.parse import urlparse

## 📦 Imports

• `os` → File handling (check/create CSV file)  
• `socket` → Detect if a URL uses an IP address  
• `numpy` / `pandas` → Data processing and feature handling  
• `streamlit` → Builds the interactive web interface  
• `sklearn` → Machine learning algorithms  
• `urlparse` → Extracts components of a URL

In [2]:
# CSV Database Handling
CSV_DB_PATH = "malicious_urls.csv"
CSV_COLUMNS = ["url", "label"]

def normalize_url(url: str) -> str:
    url = url.strip()
    if not url.startswith(("http://", "https://")):
        url = "http://" + url
    return url

def load_csv_db() -> pd.DataFrame:
    if not os.path.exists(CSV_DB_PATH):
        empty = pd.DataFrame(columns=CSV_COLUMNS)
        empty.to_csv(CSV_DB_PATH, index=False)
        return empty
    peek = pd.read_csv(CSV_DB_PATH, header=None, nrows=1)
    first_val = str(peek.iloc[0, 0]).strip().lower()
    has_header = first_val == "url"
    if has_header:
        df = pd.read_csv(CSV_DB_PATH)
        df.columns = [c.strip().lower() for c in df.columns]
    else:
        df = pd.read_csv(CSV_DB_PATH, header=None, names=["url", "label"])
    df["url"] = df["url"].astype(str).apply(normalize_url)
    df["label"] = pd.to_numeric(df["label"], errors="coerce").astype(int)
    return df[CSV_COLUMNS]

def is_in_csv_db(url: str, label: int = 1) -> bool:
    url = normalize_url(url)
    df = load_csv_db()
    match = df[df["url"].str.lower() == url.lower()]
    if match.empty: return False
    return int(match.iloc[0]["label"]) == label

## 📁 CSV Database Handling

• Stores dataset in CSV file with:
  • `url` → Website link 
  • `label` → `0` (Safe), `1` (Suspicious)  

• `normalize_url` → Ensures consistent URL format  
• `load_csv_db` → Loads dataset or creates file if missing    
• `is_in_csv_db` → Checks if URL exists  

In [3]:
# Feature Extraction
SUSPICIOUS_KEYWORDS = ["login", "verify", "bank", "secure", "update",
                       "account", "confirm", "password", "billing", "suspend"]

def contains_ip(url: str) -> int:
    try:
        host = urlparse(url).hostname or ""
        socket.inet_aton(host)
        return 1
    except OSError:
        return 0

def extract_features(url: str) -> dict:
    parsed = urlparse(url)
    hostname = parsed.hostname or ""
    full = url.lower()
    parts = [p for p in hostname.split(".") if p]
    num_subdomains = max(0, len(parts) - 2)
    kw_count = sum(kw in full for kw in SUSPICIOUS_KEYWORDS)
    return {
        "url_length": len(url),
        "num_dots": url.count("."),
        "has_at": int("@" in url),
        "has_dash": int("-" in hostname),
        "is_https": int(parsed.scheme == "https"),
        "suspicious_kws": kw_count,
        "has_ip": contains_ip(url),
        "num_subdomains": num_subdomains,
    }

FEATURE_COLS = ["url_length", "num_dots", "has_at", "has_dash",
                "is_https", "suspicious_kws", "has_ip", "num_subdomains"]

def features_to_array(features: dict) -> np.ndarray:
    return np.array([[features[c] for c in FEATURE_COLS]])

## 🔍 URL Feature Extraction

• Converts URL into numerical features for machine learning  

• Extracted Features:
  • `url_length` → Long URLs may be suspicious  
  • `num_dots` → Too many dots can indicate fake domains  
  • `has_at` → '@' symbol can hide actual destination  
  • `has_dash` → Hyphens often used in spoofed domains  
  • `is_https` → Secure websites use HTTPS  
  • `suspicious_kws` → Counts words like `login`, `verify`, `bank`  
  • `has_ip` → Checks if raw IP is used instead of domain  
  • `num_subdomains` → Too many may indicate phishing  

• `contains_ip` → Detects IP-based URLs  
• `extract_features` → Generates feature dictionary  
• `features_to_array` → Converts features into ML input  

In [4]:
# Model Training
def train_model():
    df = load_csv_db().dropna(subset=["url", "label"])
    if df.empty: return None, None
    X = pd.DataFrame([extract_features(u) for u in df["url"]])
    y = df["label"]
    if y.nunique() < 2 or len(df) < 6:
        clf = RandomForestClassifier(n_estimators=100, random_state=42)
        clf.fit(X, y)
        return clf, None
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    return clf, acc


## 🧠 Model Training

• Trains a `RandomForestClassifier` using dataset  

• Steps:
  • Extract features using `extract_features`  
  • Convert into structured data  
  • Train model using `fit()`  

• If dataset is small:
  • Model trains on full data (no split)  

• If dataset is large:
  • Uses `train_test_split`  
  • Evaluates using `accuracy_score`  

• Output:
  • `model` → Trained classifier  
  • `accuracy` → Performance score (if available)  

• Model learns patterns of safe and suspicious URLs

In [5]:
# Prediction & Explanation
def predict_url(model, url: str) -> tuple[str, float, dict]:
    features = extract_features(url)
    X = features_to_array(features)
    proba = model.predict_proba(X)[0]
    classes = list(model.classes_)
    risk = float(proba[classes.index(1)]) if 1 in classes else 0.0
    if is_in_csv_db(url, label=1): risk = 1.0
    elif is_in_csv_db(url, label=0): risk = 0.0
    label = "Suspicious" if risk >= 0.5 else "Safe"
    return label, risk, features

def explain(features: dict, label: str, in_db: bool = False) -> list[str]:
    if label == "Safe":
        reasons = ["No major red flags detected."]
        if in_db: reasons.insert(0, "✅ URL marked safe in database.")
        return reasons
    reasons = []
    if in_db: reasons.append("🗂️ URL is in CSV database.")
    if not features["is_https"]: reasons.append("🔓 Not using HTTPS.")
    if features["has_ip"]: reasons.append("🌐 Uses IP address.")
    if features["has_at"]: reasons.append("⚠️ '@' symbol in URL.")
    if features["has_dash"]: reasons.append("🔤 Hyphens in domain.")
    if features["suspicious_kws"] >= 2: reasons.append(f"🚨 {features['suspicious_kws']} suspicious keywords.")
    elif features["suspicious_kws"] == 1: reasons.append("⚠️ Contains a suspicious keyword.")
    if features["url_length"] > 75: reasons.append(f"📏 URL is long ({features['url_length']} chars).")
    if features["num_subdomains"] >= 3: reasons.append(f"🌲 {features['num_subdomains']} subdomains.")
    if not reasons: reasons.append("⚠️ URL resembles phishing patterns.")
    return reasons

## 🔮 Prediction & Explanation

• `predict_url`:
  • Uses `extract_features` to analyze URL  
  • Predicts probability using `predict_proba()`  
  • Assigns label:
    • Risk ≥ 50% → `Suspicious`  
    • Risk < 50% → `Safe`  
  • Checks CSV using `is_in_csv_db`  

• `explain`:
  • Provides reasons for prediction  

• Possible reasons:
  • Not using `HTTPS`  
  • Uses IP address (`has_ip`)  
  • Contains '@' symbol (`has_at`)  
  • Hyphens in domain (`has_dash`)  
  • Suspicious keywords (`suspicious_kws`)  
  • Long URL (`url_length`)  
  • Many subdomains (`num_subdomains`)

In [6]:
# Streamlit UI
def main():
    st.set_page_config(page_title="Suspicious URL Detector", page_icon="🔍", layout="centered")
    if "history" not in st.session_state: st.session_state.history = []
    if "model" not in st.session_state: st.session_state.model = None
    if "model_acc" not in st.session_state: st.session_state.model_acc = None
    if st.session_state.model is None:
        with st.spinner(f"Loading `{CSV_DB_PATH}` and training…"):
            st.session_state.model, st.session_state.model_acc = train_model()
    model = st.session_state.model
    st.markdown("## 🔍 Suspicious URL Detector")
    if st.session_state.model_acc is not None:
        st.caption(f"Model accuracy: {st.session_state.model_acc:.0%}")
    elif model is not None:
        st.caption("Model trained on all data (dataset too small for accuracy).")
    if st.button("🔄 Retrain model"):
        with st.spinner("Retraining…"):
            st.session_state.model, st.session_state.model_acc = train_model()
            model = st.session_state.model
        if model is not None:
            acc_msg = f" Accuracy: {st.session_state.model_acc:.0%}" if st.session_state.model_acc is not None else ""
            st.success(f"Retrained on {len(load_csv_db())} URLs.{acc_msg}")

## 💻 Streamlit User Interface

• Built using `streamlit`  

• Features:
  • User inputs URL using `text_input()`  
  • Displays:
    • Prediction (`Safe` / `Suspicious`)  
    • Risk score  
    • Explanation  
  • Shows model accuracy  
  • Allows retraining using `train_model()`  

• Uses `session_state` to:
  • Store model  
  • Maintain scan history